# 02-04_wikipedia_city_scraping_and_reusable_functions

Scraping Wikipedia content, extracting city metadata, and wrapping the logic into reusable scraping functions.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

### Loading the html

In [ ]:
url = "https://en.wikipedia.org/wiki/Web_scraping"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

soup_3 = BeautifulSoup(response.content, 'html.parser')
soup_3

> While we haven't yet looked into the requests library, we'll postpone delving into it today to avoid overwhelming you with too much new information. Instead, we'll explore the requests library when we start gathering weather data later in the project.

In [ ]:
print(soup_3.prettify())

### Getting the title

In [ ]:
soup_3.find("title").get_text()

### Getting the first h1 tag

In [ ]:
soup_3.find("h1").get_text()

### Getting all the h2 tags

In [ ]:
h2_tags = soup_3.find_all("h2")
h2_tags

As we have multiple tags in the list here, we need to use a loop to print them out.

In [ ]:
for h2 in h2_tags:
  print(h2.get_text())

### Selecting the `Legal Issues` text for only `India`
> **Pro tip:** If you're using Google Chrome, you can navigate to `View > Developer > Inspect elements` to access the built-in web development tools. Here, you can explore the HTML structure of the webpage directly within the browser using your mouse. This interactive approach is often more intuitive than examining the raw HTML code.

By investigating the html we can see that the closest, easy to access, tag is the heading with the CSS `id` of `"India"`.

In [ ]:
soup_3.find(id="India")

We can then use `.find_next()` to select the text.

In [ ]:
soup_3.find(id="India").find_next()

Looks like the next tag was a `span` tag, so let's specify that we want the next `p` tag.

In [ ]:
soup_3.find(id="India").find_next("p")

Now we can simply extract the text, and we have what we need

In [ ]:
soup_3.find(id="India").find_next("p").get_text()

## Challenge 2 😀

Utilise your web scraping skills to gather information about three German cities – Berlin, Hamburg, and Munich – from Wikipedia. You will start by extracting basic information: the country, the latitude and the longitude of each city and then expand to more dynamic data such as the population.

1. Scraping Basic Information

  1.1. Begin by scraping the country, the latitude and the longitude of each city from their respective Wikipedia pages:

 - Berlin: https://en.wikipedia.org/wiki/Berlin
 - Hamburg: https://en.wikipedia.org/wiki/Hamburg
 - Munich: https://en.wikipedia.org/wiki/Munich

  1.2. Once you have scraped the basic information of each city, reflect on the similarities and patterns in accessing them across the three pages. Also, analyse the URLs to identify any commonalities. Make a loop that executes once and retrieves the country, latitude, and longitude for all three cities.

2. Data Organisation

  2.1 Utilise pandas DataFrame to effectively store the extracted information. This DataFrame should have a row for each city, and columns for each type of information (cityname, country, latitude, longitude). If you feel brave, change latitude and longitude into decimal format.

  2.2 Looking ahead (optional): Create a function from the loop and DataFrame to encapsulate the scraping process. This function can be used repeatedly to fetch updated data whenever necessary. It should return a clean, properly formatted DataFrame.


## BONUS Challenge 3: Population

  3.1. Expand the scope of your data gathering by extracting the population of a city. This information changes over time, so we might need to add a timestamp.

  3.2. Organise your information in a DataFrame and wrap it in a separate function.

## BONUS Challenge 4: Global Data Scraping

  With your robust scraping skills now honed, venture beyond the confines of Germany and explore other cities around the world. While the extraction methodology for German cities may follow a consistent pattern, this may not be the case for cities from different countries. Can you make a function that returns a clean DataFrame of information for cities worldwide?

In [ ]:
# Challenge 2.1 — Scrape country, latitude, longitude for one city first (Berlin)
url = 'https://en.wikipedia.org/wiki/Berlin'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
soup_berlin = BeautifulSoup(response.text, 'html.parser')

infobox = soup_berlin.find('table', class_='infobox')

# Country — loop through rows to find the 'Country' label
for row in infobox.find_all('tr'):
    header = row.find('th')
    if header and 'Country' in header.get_text():
        country = row.find('td').get_text(strip=True)
        break

# Latitude and Longitude
lat_raw = soup_berlin.find('span', class_='latitude').get_text()
lon_raw = soup_berlin.find('span', class_='longitude').get_text()

print(f'Country: {country}')
print(f'Latitude: {lat_raw}')
print(f'Longitude: {lon_raw}')

In [ ]:
# Challenge 2.1 continued — helper function to convert DMS (degrees°minutes′seconds″) to decimal
import re

def dms_to_decimal(dms_str):
    """Convert a DMS string like 52°31′12″N to a decimal float like 52.5200"""
    parts = re.split(r'[°′″]', dms_str.strip())
    degrees = float(parts[0])
    minutes = float(parts[1]) if len(parts) > 1 and parts[1] else 0
    seconds = float(parts[2]) if len(parts) > 2 and parts[2] else 0
    direction = parts[-1].strip()  # N, S, E, W
    decimal = degrees + minutes / 60 + seconds / 3600
    if direction in ('S', 'W'):
        decimal *= -1
    return round(decimal, 6)

print(dms_to_decimal(lat_raw))  # Expected: ~52.52
print(dms_to_decimal(lon_raw))  # Expected: ~13.405

In [ ]:
def dms_to_decimal(dms_str):
    parts = re.split(r'[°′″]', dms_str.strip())
    degrees   = float(parts[0])
    minutes   = float(parts[1]) if len(parts) > 1 and parts[1] else 0
    # Only treat parts[2] as seconds if it's actually a number
    seconds   = float(parts[2]) if len(parts) > 2 and parts[2] and not parts[2].strip().isalpha() else 0
    direction = parts[-1].strip()
    decimal   = degrees + minutes / 60 + seconds / 3600
    if direction in ('S', 'W'):
        decimal *= -1
    return round(decimal, 6)

In [ ]:
cities = {
    'Berlin':  'https://en.wikipedia.org/wiki/Berlin',
    'Hamburg': 'https://en.wikipedia.org/wiki/Hamburg',
    'Munich':  'https://en.wikipedia.org/wiki/Munich'
}

city_data = []

for city_name, url in cities.items():
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.text, 'html.parser')
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        print(f'⚠️  No infobox found for {city_name}, skipping')
        continue

    # Country — loop through rows to find the 'Country' label
    for row in infobox.find_all('tr'):
        header = row.find('th')
        if header and 'Country' in header.get_text():
            country = row.find('td').get_text(strip=True)
            break

    lat_raw = soup.find('span', class_='latitude').get_text()
    lon_raw = soup.find('span', class_='longitude').get_text()

    city_data.append({
        'city':      city_name,
        'country':   country,
        'latitude':  dms_to_decimal(lat_raw),
        'longitude': dms_to_decimal(lon_raw)
    })
    print(f'✅ Scraped {city_name}')

print(city_data)

In [ ]:
# Challenge 2.2 — Store in a pandas DataFrame
df_cities = pd.DataFrame(city_data)
df_cities

In [ ]:
# Challenge 2.2 (optional) — Wrap everything in a reusable function
def scrape_german_cities(city_urls: dict) -> pd.DataFrame:
    """
    Scrapes country, latitude, and longitude for a dict of {city_name: wikipedia_url}.
    Returns a clean, properly formatted DataFrame.
    """
    rows = []
    for city_name, url in city_urls.items():
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(response.text, 'html.parser')
        infobox = soup.find('table', class_='infobox')
        if infobox is None:
            print(f'⚠️  No infobox found for {city_name}, skipping')
            continue

        for row in infobox.find_all('tr'):
            header = row.find('th')
            if header and 'Country' in header.get_text():
                country = row.find('td').get_text(strip=True)
                break

        lat_raw = soup.find('span', class_='latitude').get_text()
        lon_raw = soup.find('span', class_='longitude').get_text()

        rows.append({
            'city':      city_name,
            'country':   country,
            'latitude':  dms_to_decimal(lat_raw),
            'longitude': dms_to_decimal(lon_raw)
        })

    return pd.DataFrame(rows)

scrape_german_cities(cities)